In [10]:
class LetturaScrittura:
    def __init__(self, file_name):
        self.matrix = []  # attributo accessibile dall’esterno
        self.leggi(file_name)

    def leggi(self, file_name):
        """Legge un file XYZ e salva i dati in self.matrix"""
        with open(file_name, 'r') as f:
            lines = f.readlines()
        
        # Salta le prime due righe (numero atomi + commento)
        for line in lines[2:]:
            parts = line.strip().split()
            if len(parts) == 4:
                atomo = parts[0]
                x, y, z = map(float, parts[1:4])
                self.matrix.append([atomo, x, y, z])
        
    def testa(self, funzionale, basis_set, carica, molteplicità, solvent=None, dispersion=None):
    
        head = f"#p opt {funzionale}/{basis_set}"
    
        if solvent:
            head += f" scrf=(cpcm,solvent={solvent})"
        
        if dispersion:
            head += f" empiricaldispersion={dispersion}"
    
        head += "\n\n"
        head += "Title Card Required\n\n"
        head += f"{carica} {molteplicità}\n"
    
        return head 
    
    def scrivi_input(self, head, matrice):
    
        righe_xyz = []
    
        for atomo in matrice:
            simbolo = atomo[0]
            x = atomo[1]
            y = atomo[2]
            z = atomo[3]
        
            righe_xyz.append(f"{simbolo}  {x:.6f}  {y:.6f}  {z:.6f}")
    
        xyz = "\n".join(righe_xyz)
    
        com = head + xyz + "\n\n"   # riga vuota finale obbligatoria per Gaussian
    
        return com
        
# Uso
lettura = LetturaScrittura("../data/water.xyz")
for row in lettura.matrix:
    print(row)

# matrix è disponibile come attributo
print(lettura.matrix)

funzionale = 'M062x'
basis_set = 'def2svp'
carica = '1'
moltepl = '2'
sol = 'water'
disp = 'd3'
t = lettura.testa(funzionale, basis_set, carica, moltepl, solvent=sol, dispersion = disp) 
print(t)




['O', 0.0, 0.0, 0.11779]
['H', 0.0, 0.755453, -0.471161]
['H', 0.0, -0.755453, -0.471161]
[['O', 0.0, 0.0, 0.11779], ['H', 0.0, 0.755453, -0.471161], ['H', 0.0, -0.755453, -0.471161]]
#p opt M062x/def2svp scrf=(cpcm,solvent=water) empiricaldispersion=d3

Title Card Required

1 2



In [ ]:
import  numpy as np
import math

class WorkOnMatrix:
    def calc_dist(self, matrice, atomo1, atomo2):
        """
        Calcola la distanza euclidea tra due atomi nella matrice.
        matrice   : lista di liste [[atomo, x, y, z], ...]
        atomo1    : indice del primo atomo
        atomo2    : indice del secondo atomo
        intero    : se True ritorna la distanza come int, altrimenti float
        """
        # estrai le coordinate dei due atomi
        x1, y1, z1 = matrice[atomo1][1], matrice[atomo1][2], matrice[atomo1][3]
        x2, y2, z2 = matrice[atomo2][1], matrice[atomo2][2], matrice[atomo2][3]

        # calcolo distanza euclidea
        distanza = math.sqrt((x2-x1)**2 + (y2-y1)**2 + (z2-z1)**2)
        print(distanza)
        return distanza, atomo1, atomo2 
    
    def passi (self,atomo1, atomo2,passo,distanza):
        delta = distanza/passo
        if not isinstance(delta, int):
            print (f'non è un intero {delta}')
        return delta

    def new_row(self,matrice,distanza,delta,atomo1, atomo2):
        x1, y1, z1 = matrice[atomo1][1], matrice[atomo1][2], matrice[atomo1][3]
        x2, y2, z2 = matrice[atomo2][1], matrice[atomo2][2], matrice[atomo2][3]
        # vettore unitario
        ux, uy, uz = (x1-x2)/distanza, (y1-y2)/distanza, (z1-z2)/distanza
        # nuovo punto spostato verso coord2
        matrice[atomo1][1] = matrice[atomo1][1] + delta * ux
        matrice[atomo1][2] = matrice[atomo1][2] + delta * uy
        matrice[atomo1][3] = matrice[atomo1][3] + delta * uz

        return matrice

w = WorkOnMatrix()
d, atomo1, atomo2 = w.calc_dist(lettura.matrix, 0, 1)
delta = w.passi(atomo1,atomo2,0.4,d)
matrice = w.new_row(lettura.matrix,d,delta,atomo1,atomo2)
print(d)
print(atomo1)
print(atomo2)
print(delta)
print(matrice)



0.9579000551257945
non è un intero 2.394750137814486
0.9579000551257945
0
1
2.394750137814486
[['O', 0.0, -1.8886325000000002, 1.5901675000000002], ['H', 0.0, 0.755453, -0.471161], ['H', 0.0, -0.755453, -0.471161]]


In [11]:
com = lettura.scrivi_input(t, matrice)
print(com)

#p opt M062x/def2svp scrf=(cpcm,solvent=water) empiricaldispersion=d3

Title Card Required

1 2
O  0.000000  -1.888633  1.590168
H  0.000000  0.755453  -0.471161
H  0.000000  -0.755453  -0.471161


